Notebook that selects all articles from specified computer science categories with their titles and dumps them

In [9]:
# Specify categories
categories = ["Subfields_of_computer_science", "Artificial intelligence", "Software engineering", "Computer graphics", 
    "Computer security", "Computer architecture", "Theoretical computer science", "Computational science", "Formal methods",
    "Theory of computation", "Human-based computation", "Database theory", "Mathematical optimization", "Programming language theory",
    "Soft computing", "Human–computer interaction", "Algorithms and data structures", "Subfields of computer science",
    "Concurrency (computer science)", "Software architecture", "Social engineering (computer security)", "Virtual reality",
    "Internet mail protocols", "Markov models", "Neuroimaging", "Knowledge engineering", "Trees (data structures)", "Soft computing",
    "File sharing", "Vector graphics", "Digital humanities", "Temporal logic", "Applications of cryptography", "Digital media", "Petri nets",
    "Data management", "DARPA", "JavaScript", "Linux kernel features", "Electrical circuits", "Information visualization", "Software modeling language", 
    "Concurrency (computer science)", "Computer networking", "Privacy", "Data transmission", "Programming principles", "Raster graphics",
    "Computer network analysis", "Software frameworks"]

# Not allowed categories -> skipped
banned_categories = ["people", "scientists", "conference", "journal", "companies", "list", "software", "books", "biths", "deaths", "organizations", 
    "institutes", "institutions"]

# Not allowed strings in the article -> skipped
banned_substrings = ["alt=", "|", "[[", "{{", "http", "ed.", "eds.", "Ed.", "pp.", "Report", "Blog", "chap.", "Press", "Prentice Hall", "p.",
"Volume", "Proceedings", "Thesis", "et al.", "Workshop", "Congress", "&", "Session", "Linguistic and Philosophical Investigations", "Retrieved",
"books.google", "Seen", "Official Homepage", "Conference", "Prentice-Hall", "arXiv", "Journal", "HospiMedica", "ISO", "ACM Special Interest Group",
"USAF", "TechCrunch", "Dissertation", "Harvard Business Review", "Slides", "MTR-", "edition", "Accessed", "Presidential Directive", "eprint archive",
"Teil 1", "retrieved", "Guide", "page", "Springer Verlag", "see , .", "Microsoft Research", "Schirra"]

In [10]:
# Connect to database
import sqlite3

connection = sqlite3.connect("wikipedia.db")
cursor = connection.cursor()

In [11]:
# Collect all categories
data = []
missed_categories = set()

for category in categories:
    # Get articles matching each category
    cursor.execute("""
SELECT title, content
FROM articles
WHERE articles MATCH ?
    """, (f'"[[Category:{category}]]"',))

    # Iterate over all articles we got
    for row in cursor.fetchall():
        title = row[0]
        article = row[1]

        # Skip discussion sections
        if title.startswith("Wikipedia:"):
            continue

        # Skip templates
        if title.startswith("Template:"):
            continue

        # Skip categories but add them to list of missed categories
        if title.startswith("Category:"):
            missed_categories.add(title.split(":")[1])
        # Otherwise add data
        else:
            data.append(dict({
                "title": title,
                "article": article
            }))

In [12]:
# Add parser function that helps to get the first paragraph of each article we got
import re
import mwparserfromhell

def get_first_paragraph(text):
    text_categories =  []

    # Parse code next
    wikicode = mwparserfromhell.parse(text)

    # Collect categories
    for link in wikicode.filter_wikilinks():
        title = str(link.title).strip()
        if title.startswith("Category:"):
            text_categories.append(title[len("Category:"):])

    # Return empty string (which is then removed) if in one of the 'banned' categories
    for category in text_categories:
        if any(banned in category for banned in banned_categories):
            return ""

        # Add to missed category otherwise
        missed_categories.add(category)

    # Try to remove templates from article
    for template in list(wikicode.filter_templates()):
        try:
            wikicode.remove(template)
        except ValueError:
            return ""

    # Remove any codes
    clean_text = wikicode.strip_code().strip()

    # Split at first section header
    clean_text = clean_text.split("\n==", 1)[0]

    # Clean remaining paragraphs
    paragraphs = [p.strip() for p in clean_text.split("\n") if p.strip()]

    # Return first paragraph
    return paragraphs[0] if paragraphs else ""
    

In [13]:
# Clean the articles we got
cleaned_data = []

for item in data:
    paragraph = get_first_paragraph(item["article"])

    # Filter out super short articles
    if len(paragraph) > 20 and not item["title"].startswith("List of"):
        # Clean data
        paragraph = paragraph.replace("\xa0", "")
        paragraph = re.sub(r"\s*\(;[^)]*\)", "", paragraph)
        paragraph = re.sub(r"\s*\(pronounced[^)]*\)", "", paragraph)
        paragraph = paragraph.replace("* ", " ")
        paragraph = re.sub(r"\s+", " ", paragraph)
        paragraph = paragraph.replace("\\", "")
        paragraph = paragraph.replace(". .", ".")
        paragraph = paragraph.replace(" .", ".")
        paragraph = paragraph.replace("..", ".")

        if paragraph[len(paragraph) -1] == ":":
            pass
        elif any(banned in paragraph for banned in banned_substrings):
            pass
        else:
            cleaned_data.append({
                "title": item["title"],
                "text": paragraph
            })

In [14]:
# See how many paragraphs are left
len(cleaned_data)

2372

In [15]:
missed_categories

{'2004 establishments in Pennsylvania',
 'Architectural design',
 'Amendments to the United States Constitution',
 'Intelligence gathering disciplines',
 'Walking',
 'Video games about insects',
 'DIY culture',
 'Chinese-language computing',
 'Start-Class Computer Security articles of High-importance',
 'Swedish law',
 'Blended wing body',
 'Dystopian novels',
 '1964 births',
 'Viral videos',
 'Graph families',
 'Special effects',
 'Telemetry',
 'Internet properties established in 2016',
 'Social change',
 'Musical markup languages',
 '2010s controversies in the United States',
 '2018 in Europe',
 'Nuclear magnetic resonance',
 'B-Class Computer Security articles of Mid-importance',
 'Embroidery',
 'Trees (data structures)',
 'Video game sequels',
 'Alumni of the University of Edinburgh',
 'Static program analysis tools',
 'Sports video games',
 'AT&T subsidiaries',
 'Arts University Bournemouth',
 'Pontifical Catholic University of Rio de Janeiro alumni',
 'Software wars',
 'Companies

In [16]:
#import pickle
#with open("selected_data.bin", "wb") as f:
#    pickle.dump(cleaned_data, f)